# Part 3 - How sure are we about a hitter's number?

"How sure are we" sounds like one question, but it's really two — and they point at completely different objects. I ran into this the hard way.

1. **The BART posterior interval** (I'll call this Task A). This is v0's built-in uncertainty, and it turns out to be roughly *flat in PA*. It measures how uncertain the model is about the xwOBA surface at a given contact profile — not how much data we have on the player. Wrong tool for "how good is this hitter."
2. **A bootstrap over each player's plate appearances** (`player_ci`). This one behaves the way you'd want: it *narrows as PA grows*, like an honest confidence interval should.

Let me show both. The artifacts live in `results/task_a/` and `results/player_ci/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Task A — the model's own interval doesn't shrink with PA

If the BART interval were a real "how sure are we" band, it should get tighter for players with more plate appearances. Let me check whether it does.

In [ ]:
ta = jload("task_a/task_a_metrics.json")
print("n player-seasons:", ta["n_player_seasons"], " median PA:", ta["median_pa"])
print("median interval width:", round(ta["median_width"], 4))
print("log(width) ~ log(PA) slope:", round(ta["width_vs_pa_loglog"]["slope"], 3),
      "  (ideal 1/sqrt(PA) would be -0.5; ~0 means flat)")
print("width correlates with PA:      r =", round(ta["width_r_pa"], 3))
print("width correlates with xwOBA:   r =", round(ta["width_r_xwoba_mean"], 3), " (tracks contact profile, not sample size)")
print("Savant inside the 90% CI:      ", round(ta["savant_in_ci90_fraction"], 3))

In [ ]:
show_fig("task_a/figures/interval_width_vs_pa.png")

And there it is — interval width against PA is basically flat. The model's band is just as wide for a 100-PA player as for a 600-PA one. That is *not* what you want from a "how confident are we" band.

## player_ci — a bootstrap band that actually does shrink

So the model interval is the wrong object. Let me build the right one: resample each player's own plate appearances and read the spread off the bootstrap. If this is a genuine sampling band, its width should fall off roughly like 1/√PA.

In [ ]:
ci = jload("player_ci/ci_metrics.json")
print("log(width) ~ log(PA) slope:", round(ci["loglog_slope_width_vs_pa"]["slope"], 3), " (near the ideal -0.5)")
print("bootstrap width vs analytic SE:  r =", round(ci["corr_bootstrap_width_vs_analytic_se"], 3), " (genuine sampling band)")
# bootstrap vs model interval width, by PA-bin midpoint
rows = [{"PA (bin midpoint)": db["pa_mid"],
         "bootstrap width": round(db["median"], 3),
         "v0 model width": round(dm["median"], 3)}
        for db, dm in zip(ci["bootstrap_width_bin_medians"], ci["model_width_bin_medians"])]
pl.DataFrame(rows)

In [ ]:
show_fig("player_ci/figures/width_vs_pa_bootstrap_vs_model.png")

The two bands cross at around 400 PA. Below that, v0's flat interval is up to ~1.8× too narrow — meaning it *understates* uncertainty precisely in the small-sample regime where you care about it most.

In [ ]:
show_fig("player_ci/figures/example_player_bands.png")

## Takeaway

The right uncertainty band comes from resampling a player's PAs, not from BART. But there's a catch: that bootstrap band is centered on the player's *raw* number, so it still over-credits a hitter who got hot in a small sample. The fix is to correct the *center* — regress it toward the league until the PAs earn it. That regressed center is the true-talent estimate, and it's what Part 4 is all about.